# Recycling VQA Desktop Baseline

Qwen2.5-VL-3B + 4bit + LoRA 기반의 SSAFY 15기 2회차 재활용품 VQA notebook입니다.
스크립트 로직과 동일한 self-contained 구현을 notebook에 직접 포함합니다.

## 환경 준비

In [ ]:
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128
!pip install -q "transformers>=4.43.2,<5.0.0" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.3" datasets pillow pandas tqdm ipykernel

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('CUDA 환경을 먼저 확인하세요.')

## 설정

In [ ]:
from pathlib import Path

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
DATA_ROOT = '.'
TRAIN_CSV = 'train.csv'
DEV_CSV = 'dev.csv'
TEST_CSV = 'test.csv'
SAMPLE_SUBMISSION_CSV = 'sample_submission.csv'
IMAGE_SIZE = 384
SEEDS = [42]
N_FOLDS = 1
BATCH_SIZE = 1
SCORE_BATCH_SIZE = 16
EPOCHS = 1
LR = 1e-4
GRAD_ACCUM = 4
WARMUP_RATIO = 0.03
NUM_WORKERS = 0
AMP_DTYPE = 'bfloat16'
LOAD_IN_4BIT = torch.cuda.is_available()
USE_NEGATIVE_PROMPT_VARIANT = False
DEBUG_MODE = False
DEBUG_SAMPLE_SIZE = 64
SAVE_DIR = 'checkpoints/qwen2_5_vl_3b_lora_cv'
CHECKPOINT_ROOT_FOR_INFERENCE = SAVE_DIR
SUBMISSION_PATH = 'outputs/submission.csv'
RUN_TRAIN = False
RUN_SUBMISSION = False
OUTPUT_ROOT = Path(SAVE_DIR)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## 공용 유틸리티

In [ ]:
import json
import random
import re
from collections import Counter
from pathlib import Path
from typing import Iterable

import pandas as pd
import torch

CHOICES = ("a", "b", "c", "d")
TRAIN_COLUMNS = ("id", "path", "question", "a", "b", "c", "d", "answer")
TEST_COLUMNS = ("id", "path", "question", "a", "b", "c", "d")
DEV_ANSWER_COLUMNS = ("answer1", "answer2", "answer3", "answer4", "answer5")
DEV_COLUMNS = TEST_COLUMNS + DEV_ANSWER_COLUMNS
SAMPLE_SUBMISSION_COLUMNS = ("id", "answer")

DEFAULT_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
DEFAULT_SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one lowercase letter among a, b, c, or d. "
    "Do not provide any explanation."
)
FINAL_ANSWER_HINT = "정답은 반드시 a, b, c, d 중 하나의 소문자만 출력하세요."
NEGATIVE_QUESTION_HINT = "주의: 보기 중 사진에 없거나 해당하지 않는 대상을 고르세요."

NEGATIVE_QUESTION_PATTERNS = (
    "보이지 않는",
    "없는 것",
    "없는 것은",
    "아닌 것",
    "해당하지 않는",
)


def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def normalize_free_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text).strip().lower())


def normalize_choice(value: object) -> str | None:
    if value is None or pd.isna(value):
        return None

    normalized = normalize_free_text(str(value))
    if not normalized:
        return None
    if normalized in CHOICES:
        return normalized
    return None


def is_negative_question(question: str) -> bool:
    normalized = normalize_free_text(question)
    return any(pattern in normalized for pattern in NEGATIVE_QUESTION_PATTERNS)


def classify_question_type(question: str) -> str:
    normalized = normalize_free_text(question)
    if is_negative_question(normalized):
        return "negative"
    if any(token in normalized for token in ("어디", "장소", "위치")):
        return "location"
    if "색" in normalized or "컬러" in normalized:
        return "color"
    if any(token in normalized for token in ("몇 개", "개수", "몇 병", "몇 캔", "몇 상자")):
        return "count"
    if any(token in normalized for token in ("재질", "소재", "무슨 재질", "어떤 재질")):
        return "material"
    if any(token in normalized for token in ("재활용", "분리수거", "폐기물")):
        return "recycling"
    if any(token in normalized for token in ("무엇", "어떤", "종류")):
        return "what"
    return "other"


def build_mc_prompt(
    question: str,
    a: str,
    b: str,
    c: str,
    d: str,
    use_negative_prompt_variant: bool = False,
) -> str:
    prompt_lines = [question]
    if use_negative_prompt_variant and is_negative_question(question):
        prompt_lines.append(NEGATIVE_QUESTION_HINT)

    prompt_lines.extend(
        [
            f"(a) {a}",
            f"(b) {b}",
            f"(c) {c}",
            f"(d) {d}",
            "",
            FINAL_ANSWER_HINT,
        ]
    )
    return "\n".join(prompt_lines)


def extract_choice(text: str, options: dict[str, str] | None = None) -> str:
    normalized = normalize_free_text(text)
    if not normalized:
        return "a"

    if normalized in CHOICES:
        return normalized

    matches = re.findall(r"\b([abcd])\b", normalized)
    if matches:
        return matches[-1]

    if options:
        for choice, option_text in options.items():
            option_normalized = normalize_free_text(option_text)
            if not option_normalized:
                continue
            if normalized == option_normalized:
                return choice
            if option_normalized in normalized or normalized in option_normalized:
                return choice

    return "a"


def resolve_path(data_root: str | Path, relative_or_absolute_path: str) -> Path:
    path = Path(relative_or_absolute_path)
    if path.is_absolute():
        return path
    return Path(data_root) / path


def ensure_parent_dir(path: str | Path) -> Path:
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    return output_path


def save_json(path: str | Path, payload: dict) -> None:
    output_path = ensure_parent_dir(path)
    output_path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def load_json(path: str | Path) -> dict:
    return json.loads(Path(path).read_text(encoding="utf-8"))


def validate_columns(df: pd.DataFrame, expected_columns: Iterable[str], dataset_name: str) -> None:
    expected = list(expected_columns)
    actual = list(df.columns)
    if actual != expected:
        raise ValueError(
            f"{dataset_name} columns mismatch. expected={expected}, actual={actual}"
        )


def validate_choice_values(values: pd.Series, dataset_name: str, column_name: str) -> None:
    normalized = values.map(normalize_choice)
    invalid = sorted({str(raw) for raw, parsed in zip(values.tolist(), normalized.tolist(), strict=True) if parsed is None})
    if invalid:
        preview = ", ".join(invalid[:5])
        raise ValueError(
            f"{dataset_name}.{column_name} contains invalid choices: {preview}"
        )


def validate_choice_columns(values: pd.Series, dataset_name: str, column_name: str) -> None:
    normalized = values.map(normalize_choice)
    invalid = sorted(
        {
            str(raw)
            for raw, parsed in zip(values.tolist(), normalized.tolist(), strict=True)
            if parsed is None and not (pd.isna(raw) or normalize_free_text(str(raw)) == "")
        }
    )
    if invalid:
        preview = ", ".join(invalid[:5])
        raise ValueError(
            f"{dataset_name}.{column_name} contains invalid choices: {preview}"
        )


def validate_paths_exist(df: pd.DataFrame, data_root: str | Path, dataset_name: str) -> None:
    missing_paths: list[str] = []
    for raw_path in df["path"].astype(str).tolist():
        resolved = resolve_path(data_root, raw_path)
        if not resolved.exists():
            missing_paths.append(str(resolved))
            if len(missing_paths) >= 5:
                break

    if missing_paths:
        preview = ", ".join(missing_paths)
        raise FileNotFoundError(f"{dataset_name} has missing image paths: {preview}")


def validate_submission_template(test_df: pd.DataFrame, submission_df: pd.DataFrame, dataset_name: str) -> None:
    validate_columns(submission_df, SAMPLE_SUBMISSION_COLUMNS, dataset_name)
    if len(test_df) != len(submission_df):
        raise ValueError(
            f"{dataset_name} row count mismatch. expected={len(test_df)}, actual={len(submission_df)}"
        )

    expected_ids = test_df["id"].astype(str).tolist()
    actual_ids = submission_df["id"].astype(str).tolist()
    if expected_ids != actual_ids:
        raise ValueError(f"{dataset_name} ids do not match test.csv exactly.")


def load_train_frame(path: str | Path, data_root: str | Path) -> pd.DataFrame:
    df = pd.read_csv(path).reset_index(drop=True)
    validate_columns(df, TRAIN_COLUMNS, "train.csv")
    validate_choice_values(df["answer"], "train.csv", "answer")
    validate_paths_exist(df, data_root, "train.csv")
    return df


def load_test_frame(path: str | Path, data_root: str | Path) -> pd.DataFrame:
    df = pd.read_csv(path).reset_index(drop=True)
    validate_columns(df, TEST_COLUMNS, "test.csv")
    validate_paths_exist(df, data_root, "test.csv")
    return df


def build_dev_vote_record(answer_values: Iterable[object]) -> dict[str, object]:
    normalized_answers = [choice for choice in (normalize_choice(value) for value in answer_values) if choice is not None]
    valid_vote_count = len(normalized_answers)
    if valid_vote_count == 0:
        return {
            "answer": None,
            "label_status": "empty",
            "agreement": 0.0,
            "valid_vote_count": 0,
            "majority_vote_count": 0,
        }

    counts = Counter(normalized_answers)
    top_vote_count = max(counts.values())
    winning_choices = sorted(choice for choice, count in counts.items() if count == top_vote_count)
    label_status = "majority" if len(winning_choices) == 1 else "tie"
    answer = winning_choices[0] if label_status == "majority" else None
    return {
        "answer": answer,
        "label_status": label_status,
        "agreement": top_vote_count / valid_vote_count,
        "valid_vote_count": valid_vote_count,
        "majority_vote_count": top_vote_count,
    }


def load_dev_frame(path: str | Path, data_root: str | Path) -> pd.DataFrame:
    df = pd.read_csv(path).reset_index(drop=True)
    validate_columns(df, DEV_COLUMNS, "dev.csv")
    validate_paths_exist(df, data_root, "dev.csv")

    for column_name in DEV_ANSWER_COLUMNS:
        validate_choice_columns(df[column_name], "dev.csv", column_name)

    vote_records = [build_dev_vote_record(row) for row in df[list(DEV_ANSWER_COLUMNS)].itertuples(index=False, name=None)]
    vote_frame = pd.DataFrame(vote_records)
    dev_df = pd.concat([df, vote_frame], axis=1)
    return dev_df


def load_sample_submission_frame(
    path: str | Path,
    test_df: pd.DataFrame,
) -> pd.DataFrame:
    submission_df = pd.read_csv(path).reset_index(drop=True)
    validate_submission_template(test_df, submission_df, "sample_submission.csv")
    return submission_df


def make_sample_submission_frame(test_df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({"id": test_df["id"].astype(str), "answer": ""})


In [ ]:
from pathlib import Path

import torch
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig


def get_device() -> str:
    return "cuda" if torch.cuda.is_available() else "cpu"


def resolve_dtype(name: str) -> torch.dtype:
    mapping = {
        "float16": torch.float16,
        "bfloat16": torch.bfloat16,
        "float32": torch.float32,
    }
    if name not in mapping:
        raise ValueError(f"Unsupported dtype: {name}")
    return mapping[name]


def load_processor(model_source: str | Path, image_size: int, local_files_only: bool = False):
    return AutoProcessor.from_pretrained(
        model_source,
        min_pixels=image_size * image_size,
        max_pixels=image_size * image_size,
        local_files_only=local_files_only,
        trust_remote_code=True,
    )


def create_quantization_config(compute_dtype: torch.dtype):
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
    )


def load_base_model(
    model_id: str,
    load_in_4bit: bool,
    model_dtype: torch.dtype,
    local_files_only: bool = False,
):
    model_kwargs = {"trust_remote_code": True, "local_files_only": local_files_only}

    if load_in_4bit:
        model_kwargs["quantization_config"] = create_quantization_config(model_dtype)
        model_kwargs["device_map"] = "auto"
    else:
        model_kwargs["torch_dtype"] = model_dtype
        if torch.cuda.is_available():
            model_kwargs["device_map"] = "auto"

    return AutoModelForVision2Seq.from_pretrained(model_id, **model_kwargs)


def load_trainable_model(
    model_id: str,
    load_in_4bit: bool,
    model_dtype: torch.dtype,
    gradient_checkpointing: bool = True,
):
    model = load_base_model(
        model_id=model_id,
        load_in_4bit=load_in_4bit,
        model_dtype=model_dtype,
    )

    if load_in_4bit:
        model = prepare_model_for_kbit_training(model)

    if gradient_checkpointing:
        model.gradient_checkpointing_enable()

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        task_type="CAUSAL_LM",
    )

    return get_peft_model(model, lora_config)


def load_inference_model(
    checkpoint_dir: str | Path,
    model_id: str,
    load_in_4bit: bool,
    model_dtype: torch.dtype,
    local_files_only: bool = False,
):
    base_model = load_base_model(
        model_id=model_id,
        load_in_4bit=load_in_4bit,
        model_dtype=model_dtype,
        local_files_only=local_files_only,
    )
    return PeftModel.from_pretrained(base_model, checkpoint_dir)


In [ ]:
from dataclasses import dataclass
from typing import Any

import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset


Image.MAX_IMAGE_PIXELS = None


def build_prompt_messages(
    *,
    image: Image.Image,
    question: str,
    a: str,
    b: str,
    c: str,
    d: str,
    system_instruction: str,
    use_negative_prompt_variant: bool,
) -> list[dict[str, Any]]:
    user_text = build_mc_prompt(
        question=question,
        a=a,
        b=b,
        c=c,
        d=d,
        use_negative_prompt_variant=use_negative_prompt_variant,
    )
    return [
        {
            "role": "system",
            "content": [{"type": "text", "text": system_instruction}],
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": user_text},
            ],
        },
    ]


class VQAMultipleChoiceDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        processor: Any,
        data_root: str = ".",
        include_answers: bool = True,
        system_instruction: str = DEFAULT_SYSTEM_INSTRUCT,
        use_negative_prompt_variant: bool = False,
    ) -> None:
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.data_root = data_root
        self.include_answers = include_answers
        self.system_instruction = system_instruction
        self.use_negative_prompt_variant = use_negative_prompt_variant

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, index: int) -> dict[str, Any]:
        row = self.df.iloc[index]
        image_path = resolve_path(self.data_root, str(row["path"]))
        image = Image.open(image_path).convert("RGB")

        option_map = {
            "a": str(row["a"]),
            "b": str(row["b"]),
            "c": str(row["c"]),
            "d": str(row["d"]),
        }
        prompt_messages = build_prompt_messages(
            image=image,
            question=str(row["question"]),
            a=option_map["a"],
            b=option_map["b"],
            c=option_map["c"],
            d=option_map["d"],
            system_instruction=self.system_instruction,
            use_negative_prompt_variant=self.use_negative_prompt_variant,
        )

        messages = list(prompt_messages)
        target_text = None
        gold = None
        if self.include_answers:
            gold = normalize_choice(row.get("answer"))
            if gold is not None:
                target_text = gold
                messages = list(prompt_messages)
                messages.append(
                    {
                        "role": "assistant",
                        "content": [{"type": "text", "text": gold}],
                    }
                )

        return {
            "id": str(row["id"]),
            "image": image,
            "messages": messages,
            "prompt_messages": prompt_messages,
            "target_text": target_text,
            "gold": gold,
            "question": str(row["question"]),
            "option_map": option_map,
            "path": str(row["path"]),
            "label_status": row.get("label_status"),
            "agreement": row.get("agreement"),
            "valid_vote_count": row.get("valid_vote_count"),
            "majority_vote_count": row.get("majority_vote_count"),
        }


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch: list[dict[str, Any]]) -> dict[str, Any]:
        texts: list[str] = []
        prompt_texts: list[str] = []
        images: list[Image.Image] = []

        for sample in batch:
            prompt_text = self.processor.apply_chat_template(
                sample["prompt_messages"],
                tokenize=False,
                add_generation_prompt=True,
            )
            prompt_texts.append(prompt_text)
            images.append(sample["image"])

            if self.train:
                texts.append(f"{prompt_text}{sample['target_text']}")
            else:
                texts.append(prompt_text)

        encoded = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        if self.train:
            prompt_encoded = self.processor(
                text=prompt_texts,
                images=images,
                padding=True,
                return_tensors="pt",
            )
            labels = encoded["input_ids"].clone()
            labels[encoded["attention_mask"] == 0] = -100

            prompt_lengths = prompt_encoded["attention_mask"].sum(dim=1).tolist()
            full_lengths = encoded["attention_mask"].sum(dim=1).tolist()
            sequence_length = labels.shape[1]
            for index, (prompt_length, full_length) in enumerate(zip(prompt_lengths, full_lengths, strict=True)):
                answer_length = max(int(full_length - prompt_length), 0)
                answer_start = sequence_length - answer_length
                labels[index, :answer_start] = -100

            encoded["labels"] = labels

        return encoded


In [ ]:
from contextlib import nullcontext
from typing import Any

import pandas as pd
import torch
from tqdm.auto import tqdm



def autocast_context(device: str, amp_dtype: torch.dtype):
    if device == "cuda":
        return torch.autocast(device_type="cuda", dtype=amp_dtype)
    return nullcontext()


def compute_masked_mean_logprob(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    shift_logits = logits[:, :-1, :]
    shift_labels = labels[:, 1:]
    valid_mask = shift_labels.ne(-100)

    log_probs = torch.log_softmax(shift_logits, dim=-1)
    safe_labels = shift_labels.masked_fill(~valid_mask, 0)
    selected_log_probs = log_probs.gather(dim=-1, index=safe_labels.unsqueeze(-1)).squeeze(-1)

    token_counts = valid_mask.sum(dim=1).clamp(min=1)
    sequence_scores = (selected_log_probs * valid_mask).sum(dim=1) / token_counts
    return sequence_scores


def score_candidate_examples(
    *,
    model: Any,
    processor: Any,
    candidates: list[dict[str, Any]],
    device: str,
    amp_dtype: torch.dtype,
) -> list[float]:
    full_texts = [candidate["full_text"] for candidate in candidates]
    prompt_texts = [candidate["prompt_text"] for candidate in candidates]
    images = [candidate["image"] for candidate in candidates]

    encoded = processor(
        text=full_texts,
        images=images,
        padding=True,
        return_tensors="pt",
    )
    prompt_encoded = processor(
        text=prompt_texts,
        images=images,
        padding=True,
        return_tensors="pt",
    )

    labels = encoded["input_ids"].clone()
    labels[encoded["attention_mask"] == 0] = -100

    prompt_lengths = prompt_encoded["attention_mask"].sum(dim=1).tolist()
    full_lengths = encoded["attention_mask"].sum(dim=1).tolist()
    sequence_length = labels.shape[1]
    for index, (prompt_length, full_length) in enumerate(zip(prompt_lengths, full_lengths, strict=True)):
        answer_length = max(int(full_length - prompt_length), 0)
        answer_start = sequence_length - answer_length
        labels[index, :answer_start] = -100

    inputs = {key: value.to(device) for key, value in encoded.items()}
    labels = labels.to(device)

    with torch.no_grad():
        with autocast_context(device, amp_dtype):
            logits = model(**inputs).logits

    return compute_masked_mean_logprob(logits, labels).detach().cpu().tolist()


def predict_choice_scores(
    *,
    model: Any,
    processor: Any,
    df: pd.DataFrame,
    data_root: str,
    device: str,
    amp_dtype: torch.dtype,
    batch_size: int = 4,
    score_batch_size: int = 16,
    system_instruction: str = DEFAULT_SYSTEM_INSTRUCT,
    use_negative_prompt_variant: bool = False,
    desc: str = "Scoring",
) -> pd.DataFrame:
    include_answers = "answer" in df.columns
    dataset = VQAMultipleChoiceDataset(
        df=df,
        processor=processor,
        data_root=data_root,
        include_answers=include_answers,
        system_instruction=system_instruction,
        use_negative_prompt_variant=use_negative_prompt_variant,
    )

    records: list[dict[str, Any]] = []

    total_batches = (len(dataset) + batch_size - 1) // batch_size
    for start in tqdm(range(0, len(dataset), batch_size), total=total_batches, desc=desc, unit="batch"):
        batch_samples = [dataset[index] for index in range(start, min(start + batch_size, len(dataset)))]
        candidate_examples: list[dict[str, Any]] = []

        for sample_pos, sample in enumerate(batch_samples):
            prompt_text = processor.apply_chat_template(
                sample["prompt_messages"],
                tokenize=False,
                add_generation_prompt=True,
            )

            for choice_index, choice in enumerate(CHOICES):
                candidate_examples.append(
                    {
                        "sample_pos": sample_pos,
                        "choice_index": choice_index,
                        "choice": choice,
                        "prompt_text": prompt_text,
                        "full_text": f"{prompt_text}{choice}",
                        "image": sample["image"],
                    }
                )

        batch_scores = torch.full((len(batch_samples), len(CHOICES)), -1e9, dtype=torch.float32)
        for chunk_start in range(0, len(candidate_examples), score_batch_size):
            chunk = candidate_examples[chunk_start : chunk_start + score_batch_size]
            chunk_scores = score_candidate_examples(
                model=model,
                processor=processor,
                candidates=chunk,
                device=device,
                amp_dtype=amp_dtype,
            )
            for candidate, score in zip(chunk, chunk_scores, strict=True):
                batch_scores[candidate["sample_pos"], candidate["choice_index"]] = score

        for sample, score_row in zip(batch_samples, batch_scores.tolist(), strict=True):
            score_map = {choice: float(score_row[index]) for index, choice in enumerate(CHOICES)}
            prediction = max(score_map, key=score_map.get)
            record = {
                "id": sample["id"],
                "path": sample["path"],
                "question": sample["question"],
                "pred": prediction,
                "question_type": classify_question_type(sample["question"]),
            }
            if sample.get("label_status") is not None and not pd.isna(sample["label_status"]):
                record["label_status"] = str(sample["label_status"])
            if sample.get("agreement") is not None and not pd.isna(sample["agreement"]):
                record["agreement"] = float(sample["agreement"])
            if sample.get("valid_vote_count") is not None and not pd.isna(sample["valid_vote_count"]):
                record["valid_vote_count"] = int(sample["valid_vote_count"])
            if sample.get("majority_vote_count") is not None and not pd.isna(sample["majority_vote_count"]):
                record["majority_vote_count"] = int(sample["majority_vote_count"])
            if sample["gold"] is not None:
                record["answer"] = sample["gold"]
                record["is_correct"] = prediction == sample["gold"]
            elif sample.get("label_status") is not None:
                record["answer"] = None
            for choice, score in score_map.items():
                record[f"score_{choice}"] = score
            records.append(record)

    return pd.DataFrame(records)


def summarize_prediction_frame(prediction_df: pd.DataFrame) -> dict[str, Any]:
    summary: dict[str, Any] = {"rows": int(len(prediction_df))}
    if len(prediction_df) == 0:
        return summary

    if "label_status" in prediction_df.columns:
        label_counts = prediction_df["label_status"].fillna("unlabeled").value_counts().to_dict()
        summary["label_status_counts"] = {str(key): int(value) for key, value in label_counts.items()}

    if "agreement" in prediction_df.columns:
        agreement_series = prediction_df["agreement"].dropna()
        if len(agreement_series) > 0:
            summary["agreement"] = {
                "mean": float(agreement_series.mean()),
                "min": float(agreement_series.min()),
                "max": float(agreement_series.max()),
            }

    if "answer" not in prediction_df.columns or "is_correct" not in prediction_df.columns:
        return summary

    scored_df = prediction_df[prediction_df["is_correct"].notna()].copy()
    summary["scored_rows"] = int(len(scored_df))
    if len(scored_df) == 0:
        return summary

    overall_accuracy = float(scored_df["is_correct"].mean())
    summary["accuracy"] = overall_accuracy
    summary["by_question_type"] = {}
    grouped = scored_df.groupby("question_type", dropna=False)
    for question_type, group in grouped:
        summary["by_question_type"][str(question_type)] = {
            "rows": int(len(group)),
            "accuracy": float(group["is_correct"].mean()),
        }

    weakest_groups = (
        scored_df.groupby("question_type", dropna=False)["is_correct"]
        .mean()
        .sort_values()
        .head(2)
    )
    summary["weakest_question_types"] = [
        {"question_type": str(question_type), "accuracy": float(accuracy)}
        for question_type, accuracy in weakest_groups.items()
    ]
    return summary


In [ ]:
import argparse
from pathlib import Path

import pandas as pd
import torch

    CHOICES,
    DEFAULT_MODEL_ID,
    ensure_parent_dir,
    load_json,
    load_sample_submission_frame,
    load_test_frame,
    validate_submission_template,
)


def build_arg_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description="Run inference for the SSAFY VQA baseline model.")
    parser.add_argument("--checkpoint-dir", required=True)
    parser.add_argument("--test-csv", default="test.csv")
    parser.add_argument("--sample-submission-csv", default="sample_submission.csv")
    parser.add_argument("--data-root", default=".")
    parser.add_argument("--submission-path", default="outputs/submission.csv")
    parser.add_argument("--image-size", type=int, default=None)
    parser.add_argument("--base-model-id", default=None)
    parser.add_argument("--batch-size", type=int, default=4)
    parser.add_argument("--score-batch-size", type=int, default=16)
    parser.add_argument(
        "--use-negative-prompt-variant",
        action=argparse.BooleanOptionalAction,
        default=None,
    )
    parser.add_argument(
        "--load-in-4bit",
        action=argparse.BooleanOptionalAction,
        default=torch.cuda.is_available(),
    )
    return parser


def load_runtime_config(checkpoint_dir: Path) -> dict:
    config_path = checkpoint_dir / "run_config.json"
    if config_path.exists():
        return load_json(config_path)
    return {}


def discover_checkpoint_dirs(root: Path) -> list[Path]:
    if (root / "adapter_config.json").exists():
        return [root]
    return sorted(path.parent for path in root.rglob("adapter_config.json"))


def resolve_runtime_settings(
    *,
    checkpoint_dirs: list[Path],
    base_model_id: str | None,
    image_size: int | None,
    use_negative_prompt_variant: bool | None,
    load_in_4bit: bool,
) -> dict[str, object]:
    if not checkpoint_dirs:
        raise FileNotFoundError("No checkpoint directories were provided.")

    runtime_config = load_runtime_config(checkpoint_dirs[0])
    model_id = base_model_id or runtime_config.get("model_id") or DEFAULT_MODEL_ID
    resolved_image_size = image_size or runtime_config.get("image_size") or 384
    resolved_negative_prompt_variant = (
        runtime_config.get("use_negative_prompt_variant", False)
        if use_negative_prompt_variant is None
        else use_negative_prompt_variant
    )
    amp_dtype = resolve_dtype(runtime_config.get("amp_dtype", "bfloat16"))
    model_dtype = torch.float16 if load_in_4bit else amp_dtype
    return {
        "model_id": model_id,
        "image_size": resolved_image_size,
        "use_negative_prompt_variant": resolved_negative_prompt_variant,
        "amp_dtype": amp_dtype,
        "model_dtype": model_dtype,
    }


def load_processor_for_inference(
    *,
    checkpoint_dirs: list[Path],
    model_id: str,
    image_size: int,
):
    has_local_processor = any(
        (checkpoint_dirs[0] / filename).exists()
        for filename in ("processor_config.json", "preprocessor_config.json", "tokenizer_config.json")
    )
    processor_source = checkpoint_dirs[0] if has_local_processor else model_id
    processor = load_processor(
        processor_source,
        image_size=image_size,
        local_files_only=has_local_processor,
    )
    if hasattr(processor, "tokenizer"):
        processor.tokenizer.padding_side = "left"
    return processor


def score_checkpoint_ensemble(
    *,
    checkpoint_root: str | Path,
    df: pd.DataFrame,
    data_root: str,
    batch_size: int,
    score_batch_size: int,
    base_model_id: str | None = None,
    image_size: int | None = None,
    use_negative_prompt_variant: bool | None = None,
    load_in_4bit: bool = True,
) -> pd.DataFrame:
    device = get_device()
    if load_in_4bit and device != "cuda":
        print("CUDA is unavailable, so 4-bit loading is disabled.")
        load_in_4bit = False

    checkpoint_dirs = discover_checkpoint_dirs(Path(checkpoint_root))
    if not checkpoint_dirs:
        raise FileNotFoundError(f"No checkpoint directories found under: {checkpoint_root}")

    runtime_settings = resolve_runtime_settings(
        checkpoint_dirs=checkpoint_dirs,
        base_model_id=base_model_id,
        image_size=image_size,
        use_negative_prompt_variant=use_negative_prompt_variant,
        load_in_4bit=load_in_4bit,
    )
    processor = load_processor_for_inference(
        checkpoint_dirs=checkpoint_dirs,
        model_id=str(runtime_settings["model_id"]),
        image_size=int(runtime_settings["image_size"]),
    )

    ensemble_score_frame: pd.DataFrame | None = None
    static_columns: list[str] | None = None

    for checkpoint_dir in checkpoint_dirs:
        print(f"Scoring with checkpoint: {checkpoint_dir}")
        model = load_inference_model(
            checkpoint_dir=checkpoint_dir,
            model_id=str(runtime_settings["model_id"]),
            load_in_4bit=load_in_4bit,
            model_dtype=runtime_settings["model_dtype"],
            local_files_only=True,
        )
        if not load_in_4bit and device == "cuda":
            model = model.to(device)
        model.eval()

        prediction_df = predict_choice_scores(
            model=model,
            processor=processor,
            df=df,
            data_root=data_root,
            device=device,
            amp_dtype=runtime_settings["amp_dtype"],
            batch_size=batch_size,
            score_batch_size=score_batch_size,
            use_negative_prompt_variant=bool(runtime_settings["use_negative_prompt_variant"]),
            desc=f"Inference [{checkpoint_dir.name}]",
        )
        score_columns = [f"score_{choice}" for choice in CHOICES]

        if ensemble_score_frame is None:
            static_columns = [column for column in prediction_df.columns if column not in {"pred", "is_correct", *score_columns}]
            ensemble_score_frame = prediction_df[static_columns + score_columns].copy()
        else:
            ensemble_score_frame[score_columns] = (
                ensemble_score_frame[score_columns].to_numpy()
                + prediction_df[score_columns].to_numpy()
            )

        del model
        if device == "cuda":
            torch.cuda.empty_cache()

    assert ensemble_score_frame is not None
    assert static_columns is not None

    score_columns = [f"score_{choice}" for choice in CHOICES]
    result_df = ensemble_score_frame.copy()
    predicted_columns = result_df[score_columns].idxmax(axis=1)
    result_df["pred"] = predicted_columns.str.replace("score_", "", regex=False)
    if "answer" in result_df.columns:
        result_df["is_correct"] = result_df["pred"] == result_df["answer"]
        result_df.loc[result_df["answer"].isna(), "is_correct"] = pd.NA
    return result_df


def build_submission_frame(test_df: pd.DataFrame, prediction_df: pd.DataFrame) -> pd.DataFrame:
    submission_df = pd.DataFrame({"id": test_df["id"], "answer": prediction_df["pred"]})
    invalid_answer_count = int((~submission_df["answer"].isin(CHOICES)).sum())
    if invalid_answer_count:
        raise ValueError(f"Predictions contain {invalid_answer_count} invalid answers.")
    return submission_df


def main() -> int:
    args = build_arg_parser().parse_args()
    test_df = load_test_frame(args.test_csv, args.data_root)
    sample_submission_df = load_sample_submission_frame(args.sample_submission_csv, test_df)

    prediction_df = score_checkpoint_ensemble(
        checkpoint_root=args.checkpoint_dir,
        df=test_df,
        data_root=args.data_root,
        batch_size=args.batch_size,
        score_batch_size=args.score_batch_size,
        base_model_id=args.base_model_id,
        image_size=args.image_size,
        use_negative_prompt_variant=args.use_negative_prompt_variant,
        load_in_4bit=args.load_in_4bit,
    )
    submission_df = build_submission_frame(test_df, prediction_df)
    validate_submission_template(test_df, sample_submission_df, "sample_submission.csv")
    validate_submission_template(test_df, submission_df, "submission.csv")

    submission_path = ensure_parent_dir(args.submission_path)
    submission_df.to_csv(submission_path, index=False)
    print(f"Saved submission to: {submission_path}")
    return 0


In [ ]:
import argparse
import math
import random
from contextlib import nullcontext
from pathlib import Path
from typing import Any

import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import get_linear_schedule_with_warmup



def build_arg_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description="Train the SSAFY VQA baseline model.")
    parser.add_argument("--model-id", default=DEFAULT_MODEL_ID)
    parser.add_argument("--train-csv", default="train.csv")
    parser.add_argument("--dev-csv", default="dev.csv")
    parser.add_argument("--data-root", default=".")
    parser.add_argument("--output-dir", default="checkpoints/qwen2_5_vl_3b_lora_cv")
    parser.add_argument("--image-size", type=int, default=384)
    parser.add_argument("--train-sample-size", type=int, default=0)
    parser.add_argument("--valid-ratio", type=float, default=0.1)
    parser.add_argument("--n-folds", type=int, default=5)
    parser.add_argument("--seeds", type=int, nargs="+", default=[42])
    parser.add_argument("--batch-size", type=int, default=1)
    parser.add_argument("--score-batch-size", type=int, default=16)
    parser.add_argument("--epochs", type=int, default=2)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--grad-accum", type=int, default=4)
    parser.add_argument("--warmup-ratio", type=float, default=0.03)
    parser.add_argument("--num-workers", type=int, default=0)
    parser.add_argument("--amp-dtype", choices=["float16", "bfloat16"], default="bfloat16")
    parser.add_argument(
        "--use-negative-prompt-variant",
        action=argparse.BooleanOptionalAction,
        default=False,
    )
    parser.add_argument(
        "--load-in-4bit",
        action=argparse.BooleanOptionalAction,
        default=torch.cuda.is_available(),
    )
    return parser


def autocast_context(device: str, amp_dtype: torch.dtype):
    if device == "cuda":
        return torch.autocast(device_type="cuda", dtype=amp_dtype)
    return nullcontext()


def move_batch_to_device(batch: dict[str, torch.Tensor], device: str) -> dict[str, torch.Tensor]:
    return {key: value.to(device) for key, value in batch.items()}


def maybe_sample_dataframe(df: pd.DataFrame, sample_size: int) -> pd.DataFrame:
    if sample_size and sample_size < len(df):
        return df.iloc[:sample_size].reset_index(drop=True)
    return df.reset_index(drop=True)


def build_stratified_folds(labels: list[str], n_splits: int, seed: int) -> list[list[int]]:
    if n_splits < 2:
        raise ValueError("n-folds must be at least 2 for cross-validation.")

    label_to_indices: dict[str, list[int]] = {}
    for index, label in enumerate(labels):
        label_to_indices.setdefault(str(label), []).append(index)

    rng = random.Random(seed)
    folds: list[list[int]] = [[] for _ in range(n_splits)]
    for indices in label_to_indices.values():
        rng.shuffle(indices)
        for offset, index in enumerate(indices):
            folds[offset % n_splits].append(index)

    return [sorted(fold) for fold in folds]


def save_checkpoint(
    *,
    model: Any,
    processor: Any,
    checkpoint_dir: Path,
    payload: dict[str, Any],
) -> None:
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(checkpoint_dir)
    processor.save_pretrained(checkpoint_dir)
    save_json(checkpoint_dir / "run_config.json", payload)


def build_checkpoint_payload(
    *,
    args: argparse.Namespace,
    seed: int,
    fold_name: str,
    train_rows: int,
    valid_rows: int,
    best_epoch: int | None,
    best_valid_accuracy: float | None,
) -> dict[str, Any]:
    return {
        "model_id": args.model_id,
        "image_size": args.image_size,
        "load_in_4bit": args.load_in_4bit,
        "amp_dtype": args.amp_dtype,
        "seed": seed,
        "fold_name": fold_name,
        "train_rows": train_rows,
        "valid_rows": valid_rows,
        "epochs": args.epochs,
        "batch_size": args.batch_size,
        "grad_accum": args.grad_accum,
        "score_batch_size": args.score_batch_size,
        "use_negative_prompt_variant": args.use_negative_prompt_variant,
        "best_epoch": best_epoch,
        "best_valid_accuracy": best_valid_accuracy,
    }


def train_single_experiment(
    *,
    args: argparse.Namespace,
    seed: int,
    fold_name: str,
    train_subset: pd.DataFrame,
    valid_subset: pd.DataFrame,
    checkpoint_dir: Path,
    device: str,
    amp_dtype: torch.dtype,
    model_dtype: torch.dtype,
) -> tuple[dict[str, Any], pd.DataFrame | None]:
    set_seed(seed)
    processor = load_processor(args.model_id, image_size=args.image_size)
    if hasattr(processor, "tokenizer"):
        processor.tokenizer.padding_side = "left"

    model = load_trainable_model(
        model_id=args.model_id,
        load_in_4bit=args.load_in_4bit,
        model_dtype=model_dtype,
    )

    if not args.load_in_4bit and device == "cuda":
        model = model.to(device)

    train_loader = DataLoader(
        VQAMultipleChoiceDataset(
            train_subset,
            processor,
            data_root=args.data_root,
            include_answers=True,
            use_negative_prompt_variant=args.use_negative_prompt_variant,
        ),
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.num_workers,
        collate_fn=DataCollator(processor=processor, train=True),
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr)
    num_training_steps = max(1, args.epochs * math.ceil(len(train_loader) / args.grad_accum))
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        int(num_training_steps * args.warmup_ratio),
        num_training_steps,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

    best_accuracy = -1.0
    best_epoch: int | None = None
    best_prediction_df: pd.DataFrame | None = None

    for epoch in range(args.epochs):
        model.train()
        running_loss = 0.0
        progress = tqdm(
            train_loader,
            desc=f"{fold_name} | seed {seed} | epoch {epoch + 1} [train]",
            unit="batch",
        )
        optimizer.zero_grad(set_to_none=True)

        for step, batch in enumerate(progress, start=1):
            batch = move_batch_to_device(batch, device)
            with autocast_context(device, amp_dtype):
                outputs = model(**batch)
                loss = outputs.loss / args.grad_accum

            if scaler.is_enabled():
                scaler.scale(loss).backward()
            else:
                loss.backward()

            running_loss += loss.item()
            should_step = (step % args.grad_accum == 0) or (step == len(train_loader))
            if should_step:
                if scaler.is_enabled():
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()

                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                progress.set_postfix({"loss": f"{running_loss:.3f}"})
                running_loss = 0.0

        if len(valid_subset) == 0:
            continue

        model.eval()
        prediction_df = predict_choice_scores(
            model=model,
            processor=processor,
            df=valid_subset,
            data_root=args.data_root,
            device=device,
            amp_dtype=amp_dtype,
            batch_size=args.batch_size,
            score_batch_size=args.score_batch_size,
            use_negative_prompt_variant=args.use_negative_prompt_variant,
            desc=f"{fold_name} | seed {seed} | epoch {epoch + 1} [valid]",
        )
        summary = summarize_prediction_frame(prediction_df)
        valid_accuracy = float(summary.get("accuracy", 0.0))
        print(
            f"{fold_name} | seed {seed} | epoch {epoch + 1} | "
            f"valid accuracy: {valid_accuracy:.4f}"
        )
        if "label_status_counts" in summary:
            counts = summary["label_status_counts"]
            print(
                f"{fold_name} | valid labels -> majority={counts.get('majority', 0)}, "
                f"tie={counts.get('tie', 0)}, empty={counts.get('empty', 0)}"
            )
        if summary.get("weakest_question_types"):
            weakest = ", ".join(
                f"{item['question_type']}={item['accuracy']:.3f}"
                for item in summary["weakest_question_types"]
            )
            print(f"{fold_name} | weakest slices: {weakest}")

        if valid_accuracy > best_accuracy:
            best_accuracy = valid_accuracy
            best_epoch = epoch + 1
            best_prediction_df = prediction_df.copy()
            save_checkpoint(
                model=model,
                processor=processor,
                checkpoint_dir=checkpoint_dir,
                payload=build_checkpoint_payload(
                    args=args,
                    seed=seed,
                    fold_name=fold_name,
                    train_rows=len(train_subset),
                    valid_rows=len(valid_subset),
                    best_epoch=best_epoch,
                    best_valid_accuracy=best_accuracy,
                ),
            )

    if len(valid_subset) == 0:
        model.eval()
        save_checkpoint(
            model=model,
            processor=processor,
            checkpoint_dir=checkpoint_dir,
            payload=build_checkpoint_payload(
                args=args,
                seed=seed,
                fold_name=fold_name,
                train_rows=len(train_subset),
                valid_rows=0,
                best_epoch=None,
                best_valid_accuracy=None,
            ),
        )

    metrics = {
        "fold_name": fold_name,
        "seed": seed,
        "train_rows": len(train_subset),
        "valid_rows": len(valid_subset),
        "best_epoch": best_epoch,
        "best_valid_accuracy": best_accuracy if best_accuracy >= 0 else None,
    }
    save_json(checkpoint_dir / "metrics.json", metrics)

    del model, processor, train_loader, optimizer, scheduler, scaler
    if device == "cuda":
        torch.cuda.empty_cache()
    return metrics, best_prediction_df


def save_dev_outputs(output_dir: Path, prediction_df: pd.DataFrame) -> None:
    prediction_df.to_csv(output_dir / "dev_predictions.csv", index=False)
    save_json(output_dir / "dev_summary.json", summarize_prediction_frame(prediction_df))


def run_cross_validation(
    *,
    args: argparse.Namespace,
    train_df: pd.DataFrame,
    dev_df: pd.DataFrame,
    device: str,
    amp_dtype: torch.dtype,
    model_dtype: torch.dtype,
) -> None:
    output_root = Path(args.output_dir)
    output_root.mkdir(parents=True, exist_ok=True)
    save_json(output_root / "train_args.json", vars(args))

    for seed in args.seeds:
        seed_dir = output_root / f"seed_{seed}"
        seed_dir.mkdir(parents=True, exist_ok=True)
        labels = train_df["answer"].astype(str).tolist()
        folds = build_stratified_folds(labels, args.n_folds, seed)

        seed_oof_frames: list[pd.DataFrame] = []
        seed_metrics: list[dict[str, Any]] = []
        for fold_index, valid_indices in enumerate(folds):
            valid_index_set = set(valid_indices)
            train_indices = [index for index in range(len(train_df)) if index not in valid_index_set]

            fold_train_df = train_df.iloc[train_indices].reset_index(drop=True)
            fold_valid_df = train_df.iloc[valid_indices].reset_index(drop=True)
            fold_dir = seed_dir / f"fold_{fold_index}"
            fold_name = f"fold_{fold_index}"

            metrics, prediction_df = train_single_experiment(
                args=args,
                seed=seed,
                fold_name=fold_name,
                train_subset=fold_train_df,
                valid_subset=fold_valid_df,
                checkpoint_dir=fold_dir,
                device=device,
                amp_dtype=amp_dtype,
                model_dtype=model_dtype,
            )
            seed_metrics.append(metrics)

            if prediction_df is not None:
                prediction_df = prediction_df.copy()
                prediction_df["seed"] = seed
                prediction_df["fold"] = fold_index
                seed_oof_frames.append(prediction_df)

        if seed_oof_frames:
            oof_df = pd.concat(seed_oof_frames, ignore_index=True)
            oof_df.to_csv(seed_dir / "oof_predictions.csv", index=False)
            save_json(seed_dir / "oof_summary.json", summarize_prediction_frame(oof_df))
        save_json(seed_dir / "fold_metrics.json", {"metrics": seed_metrics})

        dev_prediction_df = score_checkpoint_ensemble(
            checkpoint_root=seed_dir,
            df=dev_df,
            data_root=args.data_root,
            batch_size=args.batch_size,
            score_batch_size=args.score_batch_size,
            base_model_id=args.model_id,
            image_size=args.image_size,
            use_negative_prompt_variant=args.use_negative_prompt_variant,
            load_in_4bit=args.load_in_4bit,
        )
        save_dev_outputs(seed_dir, dev_prediction_df)


def run_single_split_or_full_train(
    *,
    args: argparse.Namespace,
    train_df: pd.DataFrame,
    dev_df: pd.DataFrame,
    device: str,
    amp_dtype: torch.dtype,
    model_dtype: torch.dtype,
) -> None:
    output_root = Path(args.output_dir)
    output_root.mkdir(parents=True, exist_ok=True)
    save_json(output_root / "train_args.json", vars(args))

    for seed in args.seeds:
        seed_dir = output_root / f"seed_{seed}"
        seed_dir.mkdir(parents=True, exist_ok=True)
        checkpoint_dir = seed_dir / "full_train"

        metrics, prediction_df = train_single_experiment(
            args=args,
            seed=seed,
            fold_name="full_train",
            train_subset=train_df,
            valid_subset=dev_df,
            checkpoint_dir=checkpoint_dir,
            device=device,
            amp_dtype=amp_dtype,
            model_dtype=model_dtype,
        )
        save_json(seed_dir / "metrics.json", metrics)
        if prediction_df is not None:
            save_dev_outputs(seed_dir, prediction_df)


def main() -> int:
    args = build_arg_parser().parse_args()
    device = get_device()
    if args.load_in_4bit and device != "cuda":
        print("CUDA is unavailable, so 4-bit loading is disabled.")
        args.load_in_4bit = False

    amp_dtype = resolve_dtype(args.amp_dtype)
    model_dtype = torch.float16 if args.load_in_4bit else amp_dtype

    train_df = load_train_frame(args.train_csv, args.data_root)
    dev_df = load_dev_frame(args.dev_csv, args.data_root)
    train_df = train_df.sample(frac=1.0, random_state=args.seeds[0]).reset_index(drop=True)
    train_df = maybe_sample_dataframe(train_df, args.train_sample_size)

    if args.n_folds > 1:
        run_cross_validation(
            args=args,
            train_df=train_df,
            dev_df=dev_df,
            device=device,
            amp_dtype=amp_dtype,
            model_dtype=model_dtype,
        )
    else:
        run_single_split_or_full_train(
            args=args,
            train_df=train_df,
            dev_df=dev_df,
            device=device,
            amp_dtype=amp_dtype,
            model_dtype=model_dtype,
        )

    print(f"Training finished. Outputs saved under: {args.output_dir}")
    return 0


## 데이터 로드 + sanity check

In [ ]:
train_df = load_train_frame(TRAIN_CSV, DATA_ROOT)
dev_df = load_dev_frame(DEV_CSV, DATA_ROOT)
test_df = load_test_frame(TEST_CSV, DATA_ROOT)
sample_submission_df = load_sample_submission_frame(SAMPLE_SUBMISSION_CSV, test_df)

display(train_df.head())
display(dev_df.head())
display(test_df.head())
print('train rows:', len(train_df))
print('dev rows:', len(dev_df))
print('test rows:', len(test_df))
print('dev label status:')
display(dev_df['label_status'].value_counts().sort_index())
print('train answer distribution:')
display(train_df['answer'].value_counts().sort_index())
display(train_df['question'].map(classify_question_type).value_counts())

## 학습 실행

In [ ]:
train_args = argparse.Namespace(
    model_id=MODEL_ID,
    train_csv=TRAIN_CSV,
    dev_csv=DEV_CSV,
    data_root=DATA_ROOT,
    output_dir=SAVE_DIR,
    image_size=IMAGE_SIZE,
    train_sample_size=(DEBUG_SAMPLE_SIZE if DEBUG_MODE else 0),
    valid_ratio=0.1,
    n_folds=N_FOLDS,
    seeds=SEEDS,
    batch_size=BATCH_SIZE,
    score_batch_size=SCORE_BATCH_SIZE,
    epochs=EPOCHS,
    lr=LR,
    grad_accum=GRAD_ACCUM,
    warmup_ratio=WARMUP_RATIO,
    num_workers=NUM_WORKERS,
    amp_dtype=AMP_DTYPE,
    use_negative_prompt_variant=USE_NEGATIVE_PROMPT_VARIANT,
    load_in_4bit=LOAD_IN_4BIT,
)
display(train_args)

if RUN_TRAIN:
    device = get_device()
    if train_args.load_in_4bit and device != 'cuda':
        print('CUDA가 없어 4bit를 비활성화합니다.')
        train_args.load_in_4bit = False
    amp_dtype = resolve_dtype(train_args.amp_dtype)
    model_dtype = torch.float16 if train_args.load_in_4bit else amp_dtype
    shuffled_train_df = train_df.sample(frac=1.0, random_state=train_args.seeds[0]).reset_index(drop=True)
    shuffled_train_df = maybe_sample_dataframe(shuffled_train_df, train_args.train_sample_size)
    if train_args.n_folds > 1:
        run_cross_validation(args=train_args, train_df=shuffled_train_df, dev_df=dev_df, device=device, amp_dtype=amp_dtype, model_dtype=model_dtype)
    else:
        run_single_split_or_full_train(args=train_args, train_df=shuffled_train_df, dev_df=dev_df, device=device, amp_dtype=amp_dtype, model_dtype=model_dtype)
else:
    print('RUN_TRAIN=False 입니다. True로 바꾸면 학습을 시작합니다.')

## dev 평가 확인

In [ ]:
for seed in SEEDS:
    summary_path = OUTPUT_ROOT / f'seed_{seed}' / 'dev_summary.json'
    pred_path = OUTPUT_ROOT / f'seed_{seed}' / 'dev_predictions.csv'
    if not summary_path.exists():
        print(f'dev summary not found: {summary_path}')
        continue
    summary = load_json(summary_path)
    dev_pred_df = pd.read_csv(pred_path)
    print(f'===== seed {seed} =====')
    print(summary)
    display(dev_pred_df[['id', 'label_status', 'agreement', 'answer', 'pred']].head())

## 제출 추론

In [ ]:
if RUN_SUBMISSION:
    prediction_df = score_checkpoint_ensemble(
        checkpoint_root=CHECKPOINT_ROOT_FOR_INFERENCE,
        df=test_df,
        data_root=DATA_ROOT,
        batch_size=BATCH_SIZE,
        score_batch_size=SCORE_BATCH_SIZE,
        base_model_id=MODEL_ID,
        image_size=IMAGE_SIZE,
        use_negative_prompt_variant=USE_NEGATIVE_PROMPT_VARIANT,
        load_in_4bit=LOAD_IN_4BIT,
    )
    submission_df = build_submission_frame(test_df, prediction_df)
    validate_submission_template(test_df, submission_df, 'submission.csv')
    submission_df.to_csv(SUBMISSION_PATH, index=False)
    display(submission_df.head())
    print('saved:', SUBMISSION_PATH)
    print('rows:', len(submission_df), 'expected:', len(sample_submission_df))
else:
    print('RUN_SUBMISSION=False 입니다. True로 바꾸면 추론을 시작합니다.')